In [1]:
import os, re, json, pandas as pd
from scipy.stats import randint, uniform
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import CountVectorizer

# =======================
# Localizar o arquivo CSV
# =======================
paths = [
    "./datasets/phishing_transformed.csv","../datasets/phishing_transformed.csv",
    "./datasets/phishing.csv","../datasets/phishing.csv","../../datasets/phishing.csv",
    "phishing_transformed.csv","phishing.csv"
]
csv = next((p for p in paths if os.path.exists(p)), None)
if not csv:
    raise FileNotFoundError("Coloque phishing_transformed.csv ou phishing.csv em ./datasets/")

try:
    df = pd.read_csv(csv, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv, encoding="latin1")

# =======================
# Alvo (y)
# =======================
if "Email Type_Phishing Email" in df.columns:
    y = df["Email Type_Phishing Email"].astype(int)
elif "Email Type" in df.columns:
    y = df["Email Type"].astype(str).str.lower().str.contains("phishing").astype(int)
else:
    raise ValueError("Alvo de phishing não encontrado")

# =======================
# Limpeza de texto (se existir)
# =======================
def clean(s):
    s = str(s).lower()
    s = re.sub(r'[^a-zA-Z0-9áéíóúãõâêôçÁÉÍÓÚÃÕÂÊÔÇ\s]', ' ', s)
    s = re.sub(r'\b\d+\b', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

has_text = "Email Text" in df.columns
if has_text:
    df["Email Text"] = df["Email Text"].apply(clean)

# =======================
# Features (X) e ColumnTransformer
# =======================
drop_cols = [c for c in ["Email Type","Email Type_Phishing Email"] if c in df.columns]
X = df.drop(columns=drop_cols)

num_cols = X.select_dtypes(include=["number"]).columns.tolist()

transformers = []
if has_text:
    transformers.append(("text", CountVectorizer(max_features=4000, min_df=5, max_df=0.8), "Email Text"))
    if len(num_cols) > 0:
        transformers.append(("num", StandardScaler(with_mean=False), num_cols))
else:
    if len(num_cols) == 0:
        raise ValueError("Nenhuma coluna utilizável")
    transformers.append(("num", StandardScaler(with_mean=False), num_cols))

prep = ColumnTransformer(transformers, remainder="drop")

# =======================
# Avaliação por múltiplas seeds (CV 5-fold)
# =======================
SEEDS = list(range(20))
cv_results_per_seed = []

for seed in SEEDS:
    pipeline = Pipeline([
        ("prep", prep),
        ("clf", GradientBoostingClassifier(random_state=seed))
    ])

    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    scores = cross_val_score(pipeline, X, y, scoring="accuracy", cv=cv, n_jobs=-1)

    mean_acc, std_acc = scores.mean(), scores.std()
    print(f"[Seed:{seed:02d}] acc: {mean_acc:.4f} ± {std_acc:.4f}")

    cv_results_per_seed.append({
        "seed": seed,
        "accuracy_mean": mean_acc,
        "accuracy_std": std_acc
    })

# =======================
# RandomizedSearchCV (usando última seed)
# =======================
param_dist = {
    "clf__n_estimators": randint(120, 500),
    "clf__learning_rate": uniform(0.01, 0.19),
    "clf__max_depth": randint(2, 6),
    "clf__min_samples_split": randint(2, 12),
    "clf__min_samples_leaf": randint(1, 8),
    "clf__subsample": uniform(0.6, 0.4),
}

pipeline = Pipeline([
    ("prep", prep),
    ("clf", GradientBoostingClassifier(random_state=seed))
])

cv = StratifiedKFold(5, shuffle=True, random_state=seed)

rs = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    random_state=seed,
    verbose=1
)

rs.fit(X, y)
print(f"\n✅ [RandomSearch] best_acc: {rs.best_score_:.4f}")
print(f"📌 best_params: {rs.best_params_}")

# =======================
# Salvar resultados
# =======================
os.makedirs("results", exist_ok=True)

df_seeds = pd.DataFrame(cv_results_per_seed)
csv_seeds_path = "results/phishing_gbm_cv5_seeds.csv"
df_seeds.to_csv(csv_seeds_path, index=False)
print(f"💾 CSV gerado: {csv_seeds_path} (linhas: {len(df_seeds)})")

df_rs = pd.DataFrame(rs.cv_results_)
csv_rs_path = "results/phishing_gbm_randomsearch_cv5.csv"
df_rs.to_csv(csv_rs_path, index=False)
print(f"💾 CSV gerado: {csv_rs_path} (linhas: {len(df_rs)})")

best_params_path = "results/phishing_gbm_best_params.json"
with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump(rs.best_params_, f, ensure_ascii=False, indent=2)
print(f"💾 JSON gerado: {best_params_path}")


[Seed:00] acc: 0.9290 ± 0.0027
[Seed:01] acc: 0.9292 ± 0.0040
[Seed:02] acc: 0.9297 ± 0.0022
[Seed:03] acc: 0.9299 ± 0.0026
[Seed:04] acc: 0.9297 ± 0.0044
[Seed:05] acc: 0.9296 ± 0.0081
[Seed:06] acc: 0.9299 ± 0.0016
[Seed:07] acc: 0.9286 ± 0.0025
[Seed:08] acc: 0.9279 ± 0.0027
[Seed:09] acc: 0.9291 ± 0.0035
[Seed:10] acc: 0.9302 ± 0.0029
[Seed:11] acc: 0.9295 ± 0.0041
[Seed:12] acc: 0.9291 ± 0.0029
[Seed:13] acc: 0.9298 ± 0.0023
[Seed:14] acc: 0.9285 ± 0.0030
[Seed:15] acc: 0.9292 ± 0.0036
[Seed:16] acc: 0.9301 ± 0.0018
[Seed:17] acc: 0.9299 ± 0.0027
[Seed:18] acc: 0.9310 ± 0.0044
[Seed:19] acc: 0.9303 ± 0.0026
Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ [RandomSearch] best_acc: 0.9654
📌 best_params: {'clf__learning_rate': 0.1978314888441048, 'clf__max_depth': 3, 'clf__min_samples_leaf': 4, 'clf__min_samples_split': 11, 'clf__n_estimators': 455, 'clf__subsample': 0.8974318188433855}
💾 CSV gerado: results/phishing_gbm_cv5_seeds.csv (linhas: 20)
💾 CSV gerado: result